# `rmgdb` and `rmgdatabase`

This demo notebook shows how to use the SQL-wrapped version of RMG-database to access all of the various data contained within.

`standard` holds the `rmgdb` package, which specifies the actual layout of the database.
`data` holds the `rmgdatabase` package, which uses `rmgdb` to build an actual database file from the `RMG-database` Python source files.
`data` also includes a plaintext dump of `rmdatabase` into the YAML format - this is easier to read and more broadly intercompatible with other programming tools than the original Python files in `RMG-database`.
This demo won't cover using these files, but they are available.

There are five sub-databases:

 1. kinetics
 2. solvation
 3. statmech
 4. thermo
 5. transport

Each has libraries (collated data from the literature) and families (RMG-specific subsets from the libraries).

Running this notebook to access the data only requires one dependency: `pandas`.
No database setup is needed; Python includes `sqlite3` in its standard library, which runs without any additional complications.

One could also substitute `pandas` for `polars`, `narwhals`, etc. - really any library that can read from a SQL database.
Enterprising users may also see fit to just use `sqlite3` directly and avoid dependencies altogether, but this requires a more advanced understanding of writing SQL queries.

In [1]:
import pandas as pd

Some quick background to help make this notebook make sense: much of RMG-database stores data as shown below in this random example from the solvation sub-database.

```python
entry(
    index = 2,
    label = "propane",
    molecule = "CCC",
    solute = SoluteData(
        S = 0,
        B = 0,
        E = 0,
        L = 1.05,
        A = 0,
        V = 0.5313,
    ),
    shortDesc = """""",
    longDesc =
"""
From Abarahm et al., J. Chem. Soc., Perkin Trans. 2, 1994, 1777-1791,
DOI: 10.1039/P29940001777
""",
)
```

You can see that the `SoluteData` class is __nested__ inside our call to `entry`.
This is strictly forbidden in SQL databases; instead we store all of the calls to `entry` in one table, all of the calls to `SoluteData` in another, and then provide a 'lookup key' to match the two of them back up.

This process of re-joining the two tables is cumbersome and requires knowing some SQL.
To avoid that, SQL supports __views__ - these are basically just queries against the database that you can treat like regular tables.
They avoid you having to write SQL statements to 'rebuild' the totally flat version of the database.

All of the examples below use the various views to retrieve data.
You can of course directly look at the tables, but there's no need (unless you want to write your own SQL, in which case I suggest familiarizing yourself with the schema in `standard`).
The below function is not needed to actually  use `rmgdatabase`, but is included to help the demo.

In [2]:
import sqlite3

def list_all_views(database_file):
    """
    Connects to an SQLite database and lists all views.
    
    Args:
        database_file (str): The path to the SQLite database file.
    
    Returns:
        list: A list of view names.
    """
    conn = None
    try:
        # Create a database connection
        conn = sqlite3.connect(database_file)
        cursor = conn.cursor()

        # Query the sqlite_master table for views
        cursor.execute("SELECT name FROM sqlite_master WHERE type='view' ORDER BY name;")
        
        # Fetch all results
        views = cursor.fetchall()

        # Print the results
        if views:
            print(f"Views in database '{database_file}':")
            for view in views:
                print(f"- {view[0]}")
        else:
            print(f"No views found in database '{database_file}'.")
        
        # Return the list of view names
        return [view[0] for view in views]

    except sqlite3.Error as e:
        print(f"An error occurred: {e}")
        return []
    finally:
        # Close the connection
        if conn:
            conn.close()


One final note - RMG uses its own format for storing molecular structures called the 'adjacency list'.
These can be converted into more friendly formats (INCHI, SMILES) using RMG-Py.

## `transport`

Let's start by looking at what views we have:

In [3]:
transport_db = "data/rmgdatabase/transport/transport.db"
list_all_views(transport_db);


Views in database 'data/rmgdatabase/transport/transport.db':
- label_pairs_view
- transport_groups_view
- transport_libraries_view


The `libraries` view contains data from the literature, digitized into `rmgdatabase`.
The `groups` view contains the actual substructures used by RMG to make estimations for transport properties, with the `label_pairs` view showing how the rows of that table are related to one another in the tree structure.
This demo is focused on just getting data out of `rmgdatabase` - future work can look toward re-building the estimator tree and integrating with RMG-Py.

Let's open up the transport libraries:

In [4]:
pd.read_sql("""SELECT * from transport_libraries_view""", "sqlite:///" + transport_db).set_index("id")

,name,short_description,long_description,label,adjacency_list,shapeIndex,epsilon,epsilon_unit,sigma,sigma_unit,dipoleMoment,dipoleMoment_unit,polarizability,polarizability_unit,rotrelaxcollnum
id,,,,,,,,,,,,,,,
0,GRI-Mech,GRI-Mech3.0 value fo,,AR,\n1 Ar u0 p4 c0\n,0,1134.930,J/mol,3.330,angstroms,0.000,C*m,0.000,angstroms^3,0.0
1,GRI-Mech,GRI-Mech3.0 value fo,,C(T),\nmultiplicity 3\n1 C u2 p1 c0\n,0,593.655,J/mol,3.298,angstroms,0.000,C*m,0.000,angstroms^3,0.0
2,GRI-Mech,GRI-Mech3.0 value fo,,C2,"\nmultiplicity 3\n1 C u1 p0 c0 {2,T}\n2 C u1 p...",1,810.913,J/mol,3.621,angstroms,0.000,C*m,1.760,angstroms^3,4.0
3,GRI-Mech,GRI-Mech3.0 value fo,\nSame value as C2O(S).\n,C2O(T),"\nmultiplicity 3\n1 C u2 p0 c0 {2,D}\n2 C u0 p...",1,1932.290,J/mol,3.828,angstroms,0.000,C*m,0.000,angstroms^3,1.0
4,GRI-Mech,GRI-Mech3.0 value fo,\nSame Value as C2O(T).\n,C2O(S),"\n1 C u0 p1 c0 {2,D}\n2 C u0 p0 c0 {1,D} {3,D}...",1,1932.290,J/mol,3.828,angstroms,0.000,C*m,0.000,angstroms^3,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
382,NIST_Fluorine,From NIST CH2F2 mode,,CH-CFCF3,"\nmultiplicity 2\n1 F u0 p3 c0 {5,S}\n2 F u0 p...",2,3051.410,J/mol,4.900,angstroms,0.000,De,0.000,angstroms^3,0.0
383,NIST_Fluorine,From NIST CH2F2 mode,,CHFCHCF3,"\n1 F u0 p3 c0 {5,S}\n2 F u0 p3 c0 {5,S}\n3 F ...",2,3084.670,J/mol,4.900,angstroms,0.000,De,0.000,angstroms^3,0.0
384,NIST_Fluorine,From NIST CH2F2 mode,,CFCHCF3,"\nmultiplicity 2\n1 F u0 p3 c0 {5,S}\n2 F u0 p...",2,2785.350,J/mol,4.700,angstroms,2.722,De,4.631,angstroms^3,1.0


We can now see all of the columns that are available - we probably only want to use a subset of these, so here's a function to do so:

In [5]:
def read_sql(view_name, database_file, columns=None):
    return pd.read_sql(f"""SELECT {', '.join(columns) if columns else '*'} from {view_name}""", "sqlite:///" + database_file)

In [6]:
read_sql("transport_libraries_view", transport_db, columns=["id", "name", "adjacency_list", "epsilon", "sigma"]).set_index("id")

,name,adjacency_list,epsilon,sigma
id,,,,
0,GRI-Mech,\n1 Ar u0 p4 c0\n,1134.930,3.330
1,GRI-Mech,\nmultiplicity 3\n1 C u2 p1 c0\n,593.655,3.298
2,GRI-Mech,"\nmultiplicity 3\n1 C u1 p0 c0 {2,T}\n2 C u1 p...",810.913,3.621
3,GRI-Mech,"\nmultiplicity 3\n1 C u2 p0 c0 {2,D}\n2 C u0 p...",1932.290,3.828
4,GRI-Mech,"\n1 C u0 p1 c0 {2,D}\n2 C u0 p0 c0 {1,D} {3,D}...",1932.290,3.828
...,...,...,...,...
382,NIST_Fluorine,"\nmultiplicity 2\n1 F u0 p3 c0 {5,S}\n2 F u0 p...",3051.410,4.900
383,NIST_Fluorine,"\n1 F u0 p3 c0 {5,S}\n2 F u0 p3 c0 {5,S}\n3 F ...",3084.670,4.900
384,NIST_Fluorine,"\nmultiplicity 2\n1 F u0 p3 c0 {5,S}\n2 F u0 p...",2785.350,4.700


LLMs are generally _very_ good at writing small functions like this, so they are highly recommended for this application.
This demo contains a number of useful functions for loading the sub-databases, as well.

One can also just load the _entire_ view into memory with pandas, and then throw data out as needed, though this may be less efficient.
For example, here's a query that loads only a subset of the columns (renaming one of them, just for fun) with the additional requirement that `epsilon` and `adjacency_list` are present:

In [7]:
pd.read_sql("""
    SELECT name as library_name, adjacency_list, sigma, sigma_unit 
    FROM transport_libraries_view 
    WHERE epsilon IS NOT NULL AND adjacency_list IS NOT NULL
""", "sqlite:///" + transport_db)


,library_name,adjacency_list,sigma,sigma_unit
0,GRI-Mech,\n1 Ar u0 p4 c0\n,3.330,angstroms
1,GRI-Mech,\nmultiplicity 3\n1 C u2 p1 c0\n,3.298,angstroms
2,GRI-Mech,"\nmultiplicity 3\n1 C u1 p0 c0 {2,T}\n2 C u1 p...",3.621,angstroms
3,GRI-Mech,"\nmultiplicity 3\n1 C u2 p0 c0 {2,D}\n2 C u0 p...",3.828,angstroms
4,GRI-Mech,"\n1 C u0 p1 c0 {2,D}\n2 C u0 p0 c0 {1,D} {3,D}...",3.828,angstroms
...,...,...,...,...
382,NIST_Fluorine,"\nmultiplicity 2\n1 F u0 p3 c0 {5,S}\n2 F u0 p...",4.900,angstroms
383,NIST_Fluorine,"\n1 F u0 p3 c0 {5,S}\n2 F u0 p3 c0 {5,S}\n3 F ...",4.900,angstroms
384,NIST_Fluorine,"\nmultiplicity 2\n1 F u0 p3 c0 {5,S}\n2 F u0 p...",4.700,angstroms
385,NIST_Fluorine,"\nmultiplicity 2\n1 F u0 p3 c0 {5,S}\n2 F u0 p...",4.700,angstroms


And here's the same, but in Pandas:

In [8]:
df_transport_all = pd.read_sql("SELECT * FROM transport_libraries_view", "sqlite:///" + transport_db).set_index("id")
df_transport = df_transport_all[
    df_transport_all['epsilon'].notna() & 
    df_transport_all['adjacency_list'].notna()
].copy()
df_transport.rename(columns={'name': 'library_name'}, inplace=True)
df_transport[['library_name', 'adjacency_list', 'sigma', 'sigma_unit']]

,library_name,adjacency_list,sigma,sigma_unit
id,,,,
0,GRI-Mech,\n1 Ar u0 p4 c0\n,3.330,angstroms
1,GRI-Mech,\nmultiplicity 3\n1 C u2 p1 c0\n,3.298,angstroms
2,GRI-Mech,"\nmultiplicity 3\n1 C u1 p0 c0 {2,T}\n2 C u1 p...",3.621,angstroms
3,GRI-Mech,"\nmultiplicity 3\n1 C u2 p0 c0 {2,D}\n2 C u0 p...",3.828,angstroms
4,GRI-Mech,"\n1 C u0 p1 c0 {2,D}\n2 C u0 p0 c0 {1,D} {3,D}...",3.828,angstroms
...,...,...,...,...
382,NIST_Fluorine,"\nmultiplicity 2\n1 F u0 p3 c0 {5,S}\n2 F u0 p...",4.900,angstroms
383,NIST_Fluorine,"\n1 F u0 p3 c0 {5,S}\n2 F u0 p3 c0 {5,S}\n3 F ...",4.900,angstroms
384,NIST_Fluorine,"\nmultiplicity 2\n1 F u0 p3 c0 {5,S}\n2 F u0 p...",4.700,angstroms


## `thermo`

Once more, let's look at the views:

In [9]:
thermo_db = "data/rmgdatabase/thermo/thermo.db"
list_all_views(thermo_db);


Views in database 'data/rmgdatabase/thermo/thermo.db':
- label_pairs_view
- thermo_depositories_view
- thermo_groups_view
- thermo_libraries_view


Much the same story as the `transport` sub-database, with the only addition being the `depositories` view (mean for storing metadata, currently unused).

Let's focus on just the `libraries` view, since it is a bit more complicated than `transport`:

In [10]:
thermo_df = read_sql("thermo_libraries_view", thermo_db).set_index("id")
thermo_df.head(2)

,name,short_description,long_description,label,adjacency_list,Tdata_unit,Cpdata_unit,H298,H298_unit,S298,...,c1,c2,c3,c4,c5,c6,c7,poly_Tmin,poly_Tmax,poly_T_unit
id,,,,,,,,,,,,,,,,,,,,,
0,GRI-Mech3.0-N,,,C(T),\nmultiplicity 3\n1 C u2 p1 c0\n,K,cal/(mol*K),171.271,kcal/mol,37.7801,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,GRI-Mech3.0-N,,,C2H,"\nmultiplicity 2\n1 H u0 p0 c0 {2,S}\n2 C u0 p...",K,cal/(mol*K),135.310,kcal/mol,50.9787,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
print(thermo_df.columns)

Index(['name', 'short_description', 'long_description', 'label',
       'adjacency_list', 'Tdata_unit', 'Cpdata_unit', 'H298', 'H298_unit',
       'S298', 'S298_unit', 'Tdata_1', 'Tdata_2', 'Tdata_3', 'Tdata_4',
       'Tdata_5', 'Tdata_6', 'Tdata_7', 'Cpdata_1', 'Cpdata_2', 'Cpdata_3',
       'Cpdata_4', 'Cpdata_5', 'Cpdata_6', 'Cpdata_7', 'nasa_Tmin',
       'nasa_Tmax', 'nasa_T_unit', 'E0', 'E0_unit', 'Cp0', 'Cp0_unit', 'CpInf',
       'CpInf_unit', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'poly_Tmin',
       'poly_Tmax', 'poly_T_unit'],
      dtype='str')


Generally speaking, records in the thermo library have _either_ experimental data (e.g., H298) _or_ a NASA polynomial (e.g., NASA_Tmax).
We can select each of these in Pandas:

In [12]:
thermo_data_df = thermo_df[thermo_df['H298'].notna() & thermo_df['adjacency_list'].notna()].copy().dropna(axis='columns', how='all')
thermo_data_df.head(2)

,name,short_description,long_description,label,adjacency_list,Tdata_unit,Cpdata_unit,H298,H298_unit,S298,...,Tdata_5,Tdata_6,Tdata_7,Cpdata_1,Cpdata_2,Cpdata_3,Cpdata_4,Cpdata_5,Cpdata_6,Cpdata_7
id,,,,,,,,,,,,,,,,,,,,,
0,GRI-Mech3.0-N,,,C(T),\nmultiplicity 3\n1 C u2 p1 c0\n,K,cal/(mol*K),171.271,kcal/mol,37.7801,...,800.0,1000.0,1500.0,4.9798,4.9734,4.9715,4.9711,4.9692,4.9691,4.9742
1,GRI-Mech3.0-N,,,C2H,"\nmultiplicity 2\n1 H u0 p0 c0 {2,S}\n2 C u0 p...",K,cal/(mol*K),135.310,kcal/mol,50.9787,...,800.0,1000.0,1500.0,10.0485,10.5393,10.8828,11.1958,11.9369,12.6543,14.1032


There is an additional step for the NASA polynomials: each species can (and usually _does_) have multiple NSA polynomials that are valid at different temperature ranges.
One may wish to simply group these together, or perhaps select only the polynomials valid at a certain temperature.

See the [RMG docs](https://reactionmechanismgenerator.github.io/RMG-Py/reference/thermo/nasa.html) for more information about NASA estimations.

In [13]:
thermo_nasa_df = thermo_df[thermo_df['nasa_Tmin'].notna() & thermo_df['adjacency_list'].notna()].copy().dropna(axis='columns', how='all')
thermo_nasa_df.head(5)

,name,short_description,long_description,label,adjacency_list,nasa_Tmin,nasa_Tmax,nasa_T_unit,E0,E0_unit,...,c1,c2,c3,c4,c5,c6,c7,poly_Tmin,poly_Tmax,poly_T_unit
id,,,,,,,,,,,,,,,,,,,,,
51,USC-Mech-ii,120186,\n120186\nLow T polynomial Tmin changed from 3...,AR,\n1 Ar u0 p4 c0\n,298.0,5000.0,K,NaN,NaN,...,2.50000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,-745.375,4.366000,298.0,1000.0,K
51,USC-Mech-ii,120186,\n120186\nLow T polynomial Tmin changed from 3...,AR,\n1 Ar u0 p4 c0\n,298.0,5000.0,K,NaN,NaN,...,2.50000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,-745.375,4.366000,1000.0,5000.0,K
52,USC-Mech-ii,121286,\n121286\nLow T polynomial Tmin changed from 3...,N2,"\n1 N u0 p1 c0 {2,T}\n2 N u0 p1 c0 {1,T}\n",298.0,5000.0,K,NaN,NaN,...,2.92664,1.487980e-03,-5.684760e-07,1.009700e-10,-6.753350e-15,-922.798,5.980530,1000.0,5000.0,K
52,USC-Mech-ii,121286,\n121286\nLow T polynomial Tmin changed from 3...,N2,"\n1 N u0 p1 c0 {2,T}\n2 N u0 p1 c0 {1,T}\n",298.0,5000.0,K,NaN,NaN,...,3.29868,1.408240e-03,-3.963220e-06,5.641520e-09,-2.444850e-12,-1020.900,3.950370,298.0,1000.0,K
53,USC-Mech-ii,L 7/88,\nL 7/88.\n[H]\nImported from USC-Mech ii ther...,H,\nmultiplicity 2\n1 H u1 p0 c0\n,200.0,3500.0,K,NaN,NaN,...,2.50000,-2.308430e-11,1.615620e-14,-4.735150e-18,4.981970e-22,25473.700,-0.446683,1000.0,3500.0,K


Use the `id` column for grouping together the multiple polynomials per species:

In [14]:
for id, sub_df in thermo_nasa_df.groupby('id'):
    print(sub_df)
    break

           name short_description  \
id                                  
51  USC-Mech-ii            120186   
51  USC-Mech-ii            120186   

                                     long_description label  \
id                                                            
51  \n120186\nLow T polynomial Tmin changed from 3...    AR   
51  \n120186\nLow T polynomial Tmin changed from 3...    AR   

       adjacency_list  nasa_Tmin  nasa_Tmax nasa_T_unit  E0 E0_unit  ...   c1  \
id                                                                   ...        
51  \n1 Ar u0 p4 c0\n      298.0     5000.0           K NaN     NaN  ...  2.5   
51  \n1 Ar u0 p4 c0\n      298.0     5000.0           K NaN     NaN  ...  2.5   

     c2   c3   c4   c5       c6     c7  poly_Tmin  poly_Tmax  poly_T_unit  
id                                                                         
51  0.0  0.0  0.0  0.0 -745.375  4.366      298.0     1000.0            K  
51  0.0  0.0  0.0  0.0 -745.375  4.366     10

Use filters to select at specific temperatures, e.g., between 1000 and 2000 K:

In [15]:
thermo_nasa_df[(thermo_nasa_df['poly_Tmin'] >= 1000) & (thermo_nasa_df['poly_Tmax'] <= 2000)].head(5)

,name,short_description,long_description,label,adjacency_list,nasa_Tmin,nasa_Tmax,nasa_T_unit,E0,E0_unit,...,c1,c2,c3,c4,c5,c6,c7,poly_Tmin,poly_Tmax,poly_T_unit
id,,,,,,,,,,,,,,,,,,,,,
141,USC-Mech-ii,T12/89,\nT12/89\nLow T polynomial Tmin changed from 3...,C5H5,"\nmultiplicity 2\n1 C u1 p0 c0 {2,S} {5,S} {6...",298.0,2000.0,K,NaN,NaN,...,7.47439,0.016013,-6.482310e-09,-3.581970e-09,9.236510e-13,28086.00,-16.13300,1000.0,2000.0,K
737,SulfurHaynes,Leeds,,HSO2,"\nmultiplicity 2\n1 S u1 p0 c0 {2,D} {3,D} {4,...",298.0,2000.0,K,NaN,NaN,...,1.56274,0.020691,-2.311210e-05,1.267020e-08,-2.727420e-12,-18214.80,17.55680,1000.0,2000.0,K
738,SulfurHaynes,Leeds,,HOSO,"\nmultiplicity 2\n1 O u0 p2 c0 {2,S} {4,S}\n2 ...",298.0,2000.0,K,NaN,NaN,...,9.60147,-0.025359,6.768290e-05,-6.349540e-08,1.958940e-11,-31254.00,-15.67410,1000.0,2000.0,K
740,SulfurHaynes,Leeds,,HOSO2,"\nmultiplicity 2\n1 S u0 p1 c0 {2,S} {3,D} {4,...",298.0,2000.0,K,NaN,NaN,...,7.62277,-0.004199,3.520550e-05,-4.127150e-08,1.400070e-11,-46947.80,-7.80788,1000.0,2000.0,K
748,SulfurHaynes,,"\nH298 taken from P.A. Denis, Chem. Phys. Lett...",HSO,"\nmultiplicity 2\n1 S u1 p1 c0 {2,S} {3,D}\n2 ...",298.0,2000.0,K,NaN,NaN,...,3.27129,0.005450,-3.737790e-06,1.300210e-09,-1.831140e-13,-3808.55,9.02815,1000.0,2000.0,K


## `solvation`

In [16]:
solvation_db = "data/rmgdatabase/solvation/solvation.db"
list_all_views(solvation_db);

Views in database 'data/rmgdatabase/solvation/solvation.db':
- label_pairs_view
- solute_groups_view
- solute_libraries_view
- solvent_libraries_view


This is also largely the same as the previous sub-databases, except that the libraries are split into two different views: solute and solvent.
There isn't any fundamental architectural reason for this, more just convenience for loading the two separately:

In [17]:
read_sql("solute_libraries_view", solvation_db).set_index("id")

,name,short_description,long_description,label,molecule,S,B,E,L,A,V
id,,,,,,,,,,,
0,solute,,"\nFrom Abarahm et al., J. Chem. Soc., Perkin T...",methane,C,0.000000,0.000000,0.000000,-0.323000,0.000000,0.2495
1,solute,,"\nFrom Abarahm et al., J. Chem. Soc., Perkin T...",ethane,CC,0.000000,0.000000,0.000000,0.492000,0.000000,0.3904
2,solute,,"\nFrom Abarahm et al., J. Chem. Soc., Perkin T...",propane,CCC,0.000000,0.000000,0.000000,1.050000,0.000000,0.5313
3,solute,,"\nFrom Abarahm et al., J. Chem. Soc., Perkin T...",n-butane,CCCC,0.000000,0.000000,0.000000,1.615000,0.000000,0.6722
4,solute,,"\nFrom Abarahm et al., J. Chem. Soc., Perkin T...",2-methylpropane,CC(C)C,0.000000,0.000000,0.000000,1.409000,0.000000,0.6722
...,...,...,...,...,...,...,...,...,...,...,...
445,solute,COSMO fit,\nGeometries from LithiumPrimaryThermo library...,[CH2]C#N,[CH2]C#N,0.722629,0.275263,0.360986,1.656567,0.086384,0.3827
446,solute,COSMO fit,\nGeometries from LithiumPrimaryThermo library...,[Li]N=[C]C,[Li]N=[C]C,1.451317,-0.278358,-1.267313,-2.600696,0.777030,0.5609
447,solute,COSMO fit,\nGeometries from LithiumPrimaryThermo library...,[Li]N=CC,[Li]N=CC,2.391183,1.099293,5.293384,10.955795,1.180529,0.5824


The solvent table has many more columns than the solutes:

In [18]:
df = read_sql("solvent_libraries_view", solvation_db).set_index("id")
df.columns

Index(['name', 'short_description', 'long_description', 'label', 'molecule',
       's_g', 'b_g', 'e_g', 'l_g', 'a_g', 'c_g', 's_h', 'b_h', 'e_h', 'l_h',
       'a_h', 'c_h', 'A', 'B', 'C', 'D', 'E', 'alpha', 'beta', 'eps', 'n',
       'name_in_coolprop', 'dGsolvCount', 'dGsolvMAE_val', 'dGsolvMAE_unit',
       'dHsolvCount', 'dHsolvMAE_val', 'dHsolvMAE_unit'],
      dtype='str')

In [19]:
df.head(5)

,name,short_description,long_description,label,molecule,s_g,b_g,e_g,l_g,a_g,...,beta,eps,n,name_in_coolprop,dGsolvCount,dGsolvMAE_val,dGsolvMAE_unit,dHsolvCount,dHsolvMAE_val,dHsolvMAE_unit
id,,,,,,,,,,,,,,,,,,,,,
0,solvent,,\nAbraham and Mintz parameters: fitted by Chun...,water,O,2.74983,4.84491,0.83346,-0.22544,3.92725,...,0.38,80.4,1.33300,water,5224.0,0.17,kcal/mol,58.0,1.04,kcal/mol
1,solvent,,"\nalpha = 0.328, #primary alcohols\nbeta = 0.4...",1-octanol,CCCCCCCCO,0.71369,1.42785,0.01254,0.85312,3.52275,...,0.45,10.3,1.42050,NaN,4189.0,0.21,kcal/mol,164.0,0.50,kcal/mol
2,solvent,,\nAbraham and Mintz parameters: fitted by Chun...,benzene,C1=CC=CC=C1,1.07490,0.17492,-0.32585,1.01356,0.56683,...,0.14,2.3,1.50110,benzene,110.0,0.19,kcal/mol,200.0,0.35,kcal/mol
3,solvent,,\nAbraham and Mintz parameters: fitted by Chun...,cyclohexane,C1CCCCC1,0.00000,-0.03443,-0.32662,1.03470,0.00000,...,0.00,2.0,1.42662,CycloHexane,122.0,0.22,kcal/mol,226.0,0.31,kcal/mol
4,solvent,,\nAbraham and Mintz parameters: fitted by Chun...,dibutylether,CCCCOCCCC,0.63588,-0.27257,-0.36266,0.98243,2.44885,...,0.45,3.1,1.39920,NaN,90.0,0.20,kcal/mol,78.0,0.27,kcal/mol


All of which can be filtered against, as done previously:

In [20]:
df[df['eps'].notna()][["label", "molecule", "eps"]].head(5)

,label,molecule,eps
id,,,
0,water,O,80.4
1,1-octanol,CCCCCCCCO,10.3
2,benzene,C1=CC=CC=C1,2.3
3,cyclohexane,C1CCCCC1,2.0
4,dibutylether,CCCCOCCCC,3.1


Notice that for these views, in this sub-database only, `rmgdatabase` does not use RMG's adjacency list format.
This is because `RMG-database`, for this data only, natively stores the structures as SMILES.

## `statmech`

Starting with the views:

In [21]:
statmech_db = "data/rmgdatabase/statmech/statmech.db"
list_all_views(statmech_db)

Views in database 'data/rmgdatabase/statmech/statmech.db':
- label_pairs_view
- statmech_groups_view
- statmech_libraries_view


['label_pairs_view', 'statmech_groups_view', 'statmech_libraries_view']

This is another very standard sub-database!

In [22]:
df = read_sql("statmech_libraries_view", statmech_db).set_index("id")
df.head(3)

,name,short_description,long_description,label,adjacency_list,energy,energy_unit,spin_multiplicity,optical_isomers,mass,...,harmonic_freq_3,harmonic_freq_4,harmonic_freq_5,harmonic_freq_6,harmonic_freq_7,harmonic_freq_8,harmonic_freq_9,harmonic_freq_10,harmonic_freq_11,harmonic_freq_12
id,,,,,,,,,,,,,,,,,,,,,
0,halogens_G4,B3LYP/GTBas3,,HF,"\n1 F u0 p3 c0 {2,S}\n2 H u0 p0 c0 {1,S}\n",-282.3080,kJ/mol,None,None,20.0062,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,halogens_G4,B3LYP/GTBas3,,HBr,"\n1 Br u0 p3 c0 {2,S}\n2 H u0 p0 c0 {1,S}\n",-42.7435,kJ/mol,None,None,79.9262,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,halogens_G4,B3LYP/GTBas3,,HCl,"\n1 Cl u0 p3 c0 {2,S}\n2 H u0 p0 c0 {1,S}\n",-99.1327,kJ/mol,None,None,35.9767,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


One important division in these data is the linear vs non-linear rotors - similar to the `thermo` databases, each of these can be selected separately by looking at their corresponding fields:

In [23]:
statmech_linear_df = df[df['linear_symmetry'].notna()].copy().dropna(axis='columns', how='all')
statmech_linear_df.head(5)


,name,short_description,long_description,label,adjacency_list,energy,energy_unit,mass,mass_unit,linear_inertia,linear_inertia_unit,linear_symmetry,harmonic_freq_unit,harmonic_freq_1,harmonic_freq_2,harmonic_freq_3,harmonic_freq_4,harmonic_freq_5,harmonic_freq_6,harmonic_freq_7
id,,,,,,,,,,,,,,,,,,,,
0,halogens_G4,B3LYP/GTBas3,,HF,"\n1 F u0 p3 c0 {2,S}\n2 H u0 p0 c0 {1,S}\n",-282.30800,kJ/mol,20.0062,amu,0.809097,amu*angstrom^2,1.0,cm^-1,4113.430,NaN,NaN,NaN,NaN,NaN,NaN
1,halogens_G4,B3LYP/GTBas3,,HBr,"\n1 Br u0 p3 c0 {2,S}\n2 H u0 p0 c0 {1,S}\n",-42.74350,kJ/mol,79.9262,amu,2.014440,amu*angstrom^2,1.0,cm^-1,2635.590,NaN,NaN,NaN,NaN,NaN,NaN
2,halogens_G4,B3LYP/GTBas3,,HCl,"\n1 Cl u0 p3 c0 {2,S}\n2 H u0 p0 c0 {1,S}\n",-99.13270,kJ/mol,35.9767,amu,1.614060,amu*angstrom^2,1.0,cm^-1,2956.350,NaN,NaN,NaN,NaN,NaN,NaN
3,halogens_G4,B3LYP/GTBas3,,F2,"\n1 F u0 p3 c0 {2,S}\n2 F u0 p3 c0 {1,S}\n",-5.50154,kJ/mol,37.9968,amu,18.298700,amu*angstrom^2,2.0,cm^-1,1076.600,NaN,NaN,NaN,NaN,NaN,NaN
4,halogens_G4,B3LYP/GTBas3,,FCl,"\n1 Cl u0 p3 c0 {2,S}\n2 F u0 p3 c0 {1,S}\n",-65.04860,kJ/mol,53.9673,amu,33.221000,amu*angstrom^2,1.0,cm^-1,788.172,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
statmech_nonlinear_df = df[df['linear_symmetry'].isna()].copy().dropna(axis='columns', how='all')
statmech_nonlinear_df.head(5)


,name,short_description,long_description,label,adjacency_list,energy,energy_unit,mass,mass_unit,inertia_x,...,harmonic_freq_3,harmonic_freq_4,harmonic_freq_5,harmonic_freq_6,harmonic_freq_7,harmonic_freq_8,harmonic_freq_9,harmonic_freq_10,harmonic_freq_11,harmonic_freq_12
id,,,,,,,,,,,,,,,,,,,,,
12,halogens_G4,B3LYP/GTBas3,,OF,"\n1 F u0 p3 c0 {2,S}\n2 O u0 p2 c0 {1,S} {3,S}...",-95.2653,kJ/mol,36.0011,amu,0.860315,...,3730.420,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13,halogens_G4,B3LYP/GTBas3,,OBr,"\n1 Br u0 p3 c0 {2,S}\n2 O u0 p2 c0 {1,S} {3,...",-71.8729,kJ/mol,95.9211,amu,0.831251,...,3777.610,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14,halogens_G4,B3LYP/GTBas3,,OCl,"\n1 Cl u0 p3 c0 {2,S}\n2 O u0 p2 c0 {1,S} {3,...",-84.4995,kJ/mol,51.9716,amu,0.832669,...,3767.440,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15,halogens_G4,B3LYP/GTBas3,,FOF,"\n1 F u0 p3 c0 {3,S}\n2 F u0 p3 c0 {3,S}\n3 O ...",16.0257,kJ/mol,53.9917,amu,8.341350,...,1034.580,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16,halogens_G4,B3LYP/GTBas3,,FOBr,"\n1 Br u0 p3 c0 {3,S}\n2 F u0 p3 c0 {3,S}\n3 ...",58.9402,kJ/mol,113.9120,amu,10.582500,...,907.824,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## `kinetics`

Finally, the kinetics sub-database.
This one is far and away the most complicated and the largest, so let's walk through it starting with the views:

In [25]:
kinetics_db = "data/rmgdatabase/kinetics/kinetics.db"
list_all_views(kinetics_db);

Views in database 'data/rmgdatabase/kinetics/kinetics.db':
- all_family_rules_kinetics_view
- all_family_training_kinetics_view
- all_library_kinetics_view
- kinetics_families_view
- kinetics_family_forbidden_groups_view
- kinetics_family_groups_view
- kinetics_family_training_dictionary_view
- kinetics_family_training_reaction_species_view
- kinetics_library_dictionary_view
- kinetics_library_reaction_species_view
- label_pairs_view


These are what each of these is:

 - `all_family_rules_kinetics_view`: contains the learned estimation rules for all of the kinetics families in RMG
 - `all_family_training_kinetics_view`: training reactions using to derive the rules in `all_family_rules_kinetics_view`, note that `*_family_*` views are given a `family_name` based on the RMG reaction type
 - `all_library_kinetics_view`: known reactions from the literature/quantum mechanics simulations, which are used as a starting point to derive training reactions
 - `kinetics_families_view`: the actual reaction families in RMG, including their archetypal transformation
 - `kinetics_family_forbidden_groups_view`: some reaction families forbid certain species from being included in mechanisms; this shows all of these
 - `kinetics_family_groups_view`: used in tandem with `label_pairs_view` to reconstruct the property estimation tree
 - `kinetics_family_training_dictionary_view`: maps a given family's labels into actual species adjacency lists, stored separately because of the variable number of reactants and products
 - `kinetics_family_training_reaction_species_view`: tracks which labels correspond to reactants and products within a given reaction, for a given training reaction
 - `kinetics_library_dictionary_view` and `kinetics_library_reaction_species_view`: same as the equivalent `kinetics_family_*_view`, except for libraries rather than families.

The most useful of these for data access purposes are the `all_library_kinetics_view` and `all_family_training_kinetics_view`, whereas the others are more useful from the RMG side for loading only subsets of the larger database in very specific ways.

For this demo, let's show how to pull out all of a specific reaction type like Troe and Arrhenius.

One can pull out only the columns from the database which are relevant to the kinetics type either from SQL directly, using our `read_sql` function we defined earlier:

In [26]:
df = read_sql("all_library_kinetics_view", kinetics_db, columns=["library_name", "adjacency_reaction", "overall_kinetics_type", "degeneracy", "arr_A_val", "arr_A_unit", "arr_n", "arr_Ea_val", "arr_Ea_unit", "arr_T0_val", "arr_T0_unit"])
df[df["overall_kinetics_type"] == "Arrhenius"]

,library_name,adjacency_reaction,overall_kinetics_type,degeneracy,arr_A_val,arr_A_unit,arr_n,arr_Ea_val,arr_Ea_unit,arr_T0_val,arr_T0_unit
0,C3,"multiplicity 2\n1 C u0 p0 c0 {2,D} {5,S} {6,S}...",Arrhenius,1.0,4.200000e+01,cm^3/(mol*s),3.27,11.0,kcal/mol,1.0,K
1,C3,"multiplicity 2\n1 C u0 p0 c0 {2,S} {4,D} {5,S...",Arrhenius,1.0,1.120000e+09,s^-1,0.63,27.4,kcal/mol,1.0,K
2,C3,"multiplicity 2\n1 C u0 p0 c0 {3,S} {4,S} {9,S...",Arrhenius,1.0,7.930000e+09,s^-1,1.27,31.0,kcal/mol,1.0,K
3,C3,"1 C u0 p0 c0 {2,D} {3,D}\n2 C u0 p0 c0 {1,D} {...",Arrhenius,1.0,1.860000e+01,cm^3/(mol*s),3.00,9.5,kcal/mol,1.0,K
4,C3,"multiplicity 2\n1 C u0 p0 c0 {3,S} {4,S} {7,D...",Arrhenius,1.0,1.710000e+11,s^-1,0.20,27.5,kcal/mol,1.0,K
...,...,...,...,...,...,...,...,...,...,...,...
20797,Chernov,"1 C u0 p0 c0 {2,S} {3,B} {4,B}\n2 C u0 p0 c0...",Arrhenius,1.0,2.000000e+12,cm^3/(mol*s),0.00,62608.0,J/mol,1.0,K
20798,Chernov,"1 C u0 p0 c0 {2,B} {6,B} {7,S}\n2 C u0 p0 c0...",Arrhenius,1.0,5.800000e+16,cm^3/(mol*s),-0.77,63564.1,J/mol,1.0,K
20799,Chernov,"1 C u0 p0 c0 {2,B} {3,B} {4,B}\n2 C u0 p0 c0...",Arrhenius,1.0,1.760000e+02,cm^3/(mol*s),3.25,23240.6,J/mol,1.0,K
20800,Chernov,"multiplicity 2\n1 C u0 p0 c0 {2,B} {4,S} {8,B...",Arrhenius,1.0,3.000000e+18,cm^3/(mol*s),0.00,303478.0,J/mol,1.0,K


Or instead simply load the entire dataframe and then perform column selection in Pandas:

In [27]:
df = read_sql("all_library_kinetics_view", kinetics_db)
df[df["overall_kinetics_type"] == "Troe"].dropna(axis='columns', how='all')

,library_name,reaction_id,label,adjacency_reaction,degeneracy,short_description,long_description,overall_kinetics_type,troe_alpha,troe_T3,troe_T1,troe_T2,troe_high_A,troe_high_n,troe_high_Ea,troe_low_A,troe_low_n,troe_low_Ea
1904,Klippenstein_Glarborg2016,1904,H + O2 <=> HO2,multiplicity 2\n1 H u1 p0 c0\n + \nmultiplicit...,1.0,The chemkin file reaction is H + O2 <=> HO2,,Troe,0.500,1.000000e-30,1.000000e+30,NaN,4.700000e+12,0.440,0.0,6.366000e+20,-1.720,524.8000
1925,Klippenstein_Glarborg2016,1921,H2O2 <=> OH + OH,"1 O u0 p2 c0 {2,S} {3,S}\n2 O u0 p2 c0 {1,S} {...",1.0,The chemkin file reaction is H2O2 <=> OH + OH,,Troe,0.430,1.000000e-30,1.000000e+30,NaN,2.000000e+12,0.900,48749.0,2.500000e+24,-2.300,48749.0000
1931,Klippenstein_Glarborg2016,1926,CO + O <=> CO2,"1 C u0 p1 c-1 {2,T}\n2 O u0 p1 c+1 {1,T}\n + \...",1.0,The chemkin file reaction is CO + O <=> CO2,,Troe,1.000,1.000000e-30,1.000000e+30,1.000000e+30,1.800000e+10,0.000,2384.0,1.400000e+24,-2.790,4191.0000
1937,Klippenstein_Glarborg2016,1932,HOCO <=> CO2 + H,"multiplicity 2\n1 C u1 p0 c0 {2,S} {3,D}\n2 O ...",1.0,The chemkin file reaction is HOCO <=> CO2 + H,,Troe,0.390,1.000000e-30,1.000000e+30,NaN,8.200000e+11,0.413,35335.0,6.000000e+26,-3.148,37116.0000
1963,Klippenstein_Glarborg2016,1953,HCO <=> H + CO,"multiplicity 2\n1 C u1 p0 c0 {2,S} {3,D}\n2 H ...",1.0,The chemkin file reaction is HCO <=> H + CO,,Troe,0.103,1.390000e+02,1.090000e+04,4.550000e+03,4.930000e+16,-0.930,19724.0,7.430000e+21,-2.360,19383.0000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20393,FFCM1(-),19989,C2H4 + H <=> C2H5,"1 C u0 p0 c0 {2,D} {3,S} {4,S}\n2 C u0 p0 c0 {...",1.0,,,Troe,1.569,-9.147000e+03,2.990000e+02,1.524000e+02,1.232000e+09,1.463,1355.0,2.900000e+39,-6.642,5769.0000
20404,FFCM1(-),19999,C2H5 + H <=> C2H6,"multiplicity 2\n1 C u0 p0 c0 {2,S} {3,S} {4,S}...",1.0,,,Troe,0.842,1.250000e+02,2.219000e+03,6.882000e+03,5.210000e+17,-0.990,1580.0,1.990000e+41,-7.080,6685.0000
20535,Chernov,20129,A1C2H- + H <=> A1C2H,"multiplicity 2\n1 C u0 p0 c0 {2,B} {6,B} {7,S...",1.0,,\n549\n,Troe,1.000,1.000000e-01,5.849000e+02,6.113000e+03,1.000000e+14,0.000,0.0,6.600000e+75,-16.300,58201.3000
20573,Chernov,20167,A1C2H3* + H <=> A1C2H3,"multiplicity 2\n1 C u0 p0 c0 {2,B} {4,S} {8,B...",1.0,,\n587\n,Troe,1.000,1.000000e-01,5.849000e+02,6.113000e+03,1.000000e+14,0.000,0.0,6.600000e+75,-16.300,58201.3000


`all_family_training_kinetics_view` is the same as above, except it is sorted by RMG reaction family instead of library:

In [28]:
df = read_sql("all_family_training_kinetics_view", kinetics_db)
df[df["family_name"] == "R_Recombination"].dropna(axis='columns', how='all')

,family_name,reaction_id,label,adjacency_reaction,degeneracy,short_description,long_description,rank,overall_kinetics_type,arr_A_val,arr_A_unit,arr_n,arr_Ea_val,arr_Ea_unit,arr_T0_val,arr_T0_unit
3717,R_Recombination,3717,CH3O2 <=> O2 + CH3,"multiplicity 2\n1 O u0 p2 c0 {2,S} {3,S}\n2 O ...",1.0,Method CBS-QB3 w/ 1-d Hindered rotor corrections,\nHigh-Pressure Rate Rules for Alkyl + O2 Reac...,10,Arrhenius,1.090000e+14,s^-1,0.250,33.300,kcal/mol,1.0,K
3718,R_Recombination,3718,C2H5O2 <=> O2 + C2H5,"multiplicity 2\n1 O u0 p2 c0 {2,S} {3,S}\n2 O ...",1.0,Method CBS-QB3 w/ 1-d Hindered rotor corrections,\nHigh-Pressure Rate Rules for Alkyl + O2 Reac...,10,Arrhenius,9.490000e+21,s^-1,-2.410,35.800,kcal/mol,1.0,K
3719,R_Recombination,3719,C3H7O2 <=> O2 + C3H7,"multiplicity 2\n1 O u0 p2 c0 {2,S} {4,S}\n2 ...",1.0,Method CBS-QB3 w/ 1-d Hindered rotor corrections,\nHigh-Pressure Rate Rules for Alkyl + O2 Reac...,10,Arrhenius,1.520000e+23,s^-1,-2.710,36.400,kcal/mol,1.0,K
3720,R_Recombination,3720,1-hydroxybutyl + O2 <=> 1-hydroxybutylO2,"multiplicity 2\n1 O u0 p2 c0 {5,S} {14,S}\n...",2.0,CBS-QB3 w/ 1-d HR,\nReference: Low-Temperature Combustion Chemis...,10,Arrhenius,8.360000e+12,cm^3/(mol*s),-0.085,-567.200,cal/mol,1.0,K
3721,R_Recombination,3721,NO2 + NO2 <=> N2O4,"multiplicity 2\n1 O u0 p3 c-1 {3,S}\n2 O u...",1.0,High or low pressure extrapolation,\nBath gas: N2\nExcitation technique: Flash ph...,10,Arrhenius,2.630000e+08,m^3/(mol*s),-1.100,0.000,kJ/mol,1.0,K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3887,R_Recombination,3887,H + C10H21-5 <=> C10H22-10,multiplicity 2\n1 * H u1 p0 c0\n + \nmultiplic...,1.0,,\nElectronic structure calculations carried ou...,6,Arrhenius,3.252000e+13,cm^3/(mol*s),0.212,0.006,kcal/mol,1.0,K
3888,R_Recombination,3888,CH3 + C7H15-2 <=> C8H18-2,"multiplicity 2\n1 * C u1 p0 c0 {2,S} {3,S} {4,...",1.0,,\nSpecies are optimized and calculated by the ...,6,Arrhenius,2.223000e+16,cm^3/(mol*s),-0.506,0.816,kcal/mol,1.0,K
3889,R_Recombination,3889,C4H9-3 + C4H9 <=> C8H18-3,"multiplicity 2\n1 C u0 p0 c0 {2,S} {3,S} {4...",1.0,,\nSpecies are optimized and calculated by the ...,6,Arrhenius,1.045000e+15,cm^3/(mol*s),-0.155,-1.631,kcal/mol,1.0,K
3890,R_Recombination,3890,C5H11-2 + C3H7-2 <=> C8H18-4,"multiplicity 2\n1 C u0 p0 c0 {2,S} {3,S} {4...",1.0,,\nSpecies are optimized and calculated by the ...,6,Arrhenius,3.783000e+13,cm^3/(mol*s),0.209,-1.996,kcal/mol,1.0,K


The `adjacency_reaction` columns contains the adjacency lists for each of the species, separated by the same separators as the `label`.